# Task 12.2 – American Options with Continuous Dividends including Greeks

In [1]:
import numpy as np
import pandas as pd

In [2]:
def american_binomial(S, K, T, r, q, sigma, option_type, N=500):
    """
    Price an American option with continuous dividend yield using the
    Cox-Ross-Rubinstein (CRR) binomial tree.

    Parameters
    ----------
    S : float  - Current stock price
    K : float  - Strike price
    T : float  - Time to maturity in years
    r : float  - Continuously compounded risk-free rate
    q : float  - Continuously compounded dividend yield
    sigma : float - Volatility
    option_type : str - 'Call' or 'Put'
    N : int    - Number of time steps

    Returns
    -------
    float : option price
    """
    dt = T / N
    u = np.exp(sigma * np.sqrt(dt))
    d = 1.0 / u
    p = (np.exp((r - q) * dt) - d) / (u - d)
    disc = np.exp(-r * dt)

    is_call = option_type.strip().lower() == 'call'

    # Terminal stock prices and payoffs
    j = np.arange(N + 1)
    ST = S * u ** (N - j) * d ** j
    V = np.maximum(ST - K, 0) if is_call else np.maximum(K - ST, 0)

    # Backward induction with early exercise check
    for i in range(N - 1, -1, -1):
        j = np.arange(i + 1)
        Si = S * u ** (i - j) * d ** j
        V = disc * (p * V[:i + 1] + (1 - p) * V[1:i + 2])
        intrinsic = np.maximum(Si - K, 0) if is_call else np.maximum(K - Si, 0)
        V = np.maximum(V, intrinsic)

    return V[0]


def american_greeks(S, K, T, r, q, sigma, option_type, N=500):
    """
    Compute price and Greeks for an American option via finite differences
    applied to the CRR binomial tree.

    Greeks conventions
    ------------------
    Delta : dV/dS  (bump S by ±1%)
    Gamma : d²V/dS²
    Vega  : dV/dσ  (bump σ by ±0.1 pp, result per unit σ)
    Rho   : dV/dr  (bump r by ±1 bp, result per unit r)
    Theta : dV/dT  (one-sided, bump T forward by 1 day – positive means
                    more time-to-maturity increases value)
    """
    price = american_binomial(S, K, T, r, q, sigma, option_type, N)

    # Delta / Gamma
    h_S = 0.01 * S
    V_up = american_binomial(S + h_S, K, T, r, q, sigma, option_type, N)
    V_dn = american_binomial(S - h_S, K, T, r, q, sigma, option_type, N)
    delta = (V_up - V_dn) / (2 * h_S)
    gamma = (V_up - 2 * price + V_dn) / (h_S ** 2)

    # Vega
    h_v = 0.001
    V_vu = american_binomial(S, K, T, r, q, sigma + h_v, option_type, N)
    V_vd = american_binomial(S, K, T, r, q, sigma - h_v, option_type, N)
    vega = (V_vu - V_vd) / (2 * h_v)

    # Rho
    h_r = 0.0001
    V_ru = american_binomial(S, K, T, r + h_r, q, sigma, option_type, N)
    V_rd = american_binomial(S, K, T, r - h_r, q, sigma, option_type, N)
    rho = (V_ru - V_rd) / (2 * h_r)

    # Theta: one-sided, per year  (dV/dT_remaining, positive = time has value)
    h_T = 1 / 365
    V_tp = american_binomial(S, K, T + h_T, r, q, sigma, option_type, N)
    theta = (V_tp - price) / h_T

    return {"Value": price, "Delta": delta, "Gamma": gamma,
            "Vega": vega, "Rho": rho, "Theta": theta}

In [3]:
def process_american(input_path, output_path, N=500):
    """
    Read option parameters from input_path, compute American option price
    and Greeks, print results, and save to output_path.
    """
    df = pd.read_csv(input_path).dropna(subset=["ID"])
    df["ID"] = df["ID"].astype(int)

    results = []
    for _, row in df.iterrows():
        T = row["DaysToMaturity"] / row["DayPerYear"]
        greeks = american_greeks(
            S=row["Underlying"],
            K=row["Strike"],
            T=T,
            r=row["RiskFreeRate"],
            q=row["DividendRate"],
            sigma=row["ImpliedVol"],
            option_type=row["Option Type"],
            N=N,
        )
        results.append({"ID": int(row["ID"]), **greeks})

    out = pd.DataFrame(results, columns=["ID", "Value", "Delta", "Gamma", "Vega", "Rho", "Theta"])
    print(out.to_string(index=False))
    out.to_csv(output_path, index=False)
    print(f"\nSaved to {output_path}")
    return out

results_12_2 = process_american("test12_1.csv", "testout_12.2.csv", N=500)

 ID     Value     Delta    Gamma      Vega        Rho     Theta
  1  3.259347  0.547643 0.059244 14.651750   7.058291 12.960975
  2  2.692775 -0.463483 0.060507 14.611928  -5.256947  8.887079
  3 22.050266  0.680852 0.008986 35.904831  50.209266  7.275168
  4 21.019233 -0.490073 0.002592 40.759301 -53.686420  5.951540

Saved to testout_12.2.csv
